## Classe `BaseManager`
O que representa esta classe? (responda abaixo)


### Método `BaseManager.__init__`
O que faz este método? (responda abaixo)


In [ ]:
def __init__(self, core):
        self.core = core
        log.debug(f'✓ {self.__class__.__name__} criado')

### Método `BaseManager.initialize`
O que faz este método? (responda abaixo)


In [ ]:
def initialize(self) -> bool:
        pass

### Método `BaseManager.cleanup`
O que faz este método? (responda abaixo)


In [ ]:
def cleanup(self) -> None:
        pass

### Método `BaseManager._notify`
O que faz este método? (responda abaixo)


In [ ]:
def _notify(self, event: str, data: Optional[Dict[str, Any]]=None) -> None:
        if data is None:
            data = {}
        if hasattr(self.core, '_notify'):
            try:
                self.core._notify(event, data)
            except Exception as e:
                log.warning(f"Erro ao notificar evento '{event}': {e}")

### Método `BaseManager._log`
O que faz este método? (responda abaixo)


In [ ]:
def _log(self, level: str, message: str) -> None:
        manager_name = self.__class__.__name__
        prefixed_msg = f'[{manager_name}] {message}'
        if level == 'info':
            log.info(prefixed_msg)
        elif level == 'warning':
            log.warning(prefixed_msg)
        elif level == 'error':
            log.error(prefixed_msg)
        elif level == 'debug':
            log.debug(prefixed_msg)

### Método `BaseManager.__repr__`
O que faz este método? (responda abaixo)


In [ ]:
def __repr__(self) -> str:
        return f'<{self.__class__.__name__}(core={self.core.__class__.__name__})>'

## Classe `CaptureManager`
O que representa esta classe? (responda abaixo)


### Método `CaptureManager.__init__`
O que faz este método? (responda abaixo)


In [ ]:
def __init__(self, core, camera_profiles: Optional[Any]=None):
        super().__init__(core)
        self.camera_profiles = camera_profiles
        self.current_image: Optional[np.ndarray] = None
        self.raw_image: Optional[np.ndarray] = None
        self.crop_enabled: bool = False
        self.crop_bbox: Optional[tuple] = None
        self._inspect_after_capture: Optional[str] = None
        self._log('info', 'Manager criado')

### Método `CaptureManager.initialize`
O que faz este método? (responda abaixo)


In [ ]:
def initialize(self) -> bool:
        try:
            if self.camera_profiles:
                return self.camera_profiles.load_profiles()
            return True
        except Exception as e:
            self._log('error', f'Falha na inicialização: {e}')
            return False

### Método `CaptureManager.cleanup`
O que faz este método? (responda abaixo)


In [ ]:
def cleanup(self) -> None:
        self.clear_images()
        self._log('info', 'Limpeza completa')

### Método `CaptureManager.set_crop_settings`
O que faz este método? (responda abaixo)


In [ ]:
def set_crop_settings(self, enabled: bool, bbox: Optional[tuple]=None):
        self.crop_enabled = enabled
        self.crop_bbox = bbox
        if enabled:
            self._log('info', f'Crop habilitado: {bbox}')
        else:
            self._log('info', 'Crop desabilitado')

### Método `CaptureManager.reset_crop`
O que faz este método? (responda abaixo)


In [ ]:
def reset_crop(self):
        self.crop_enabled = False
        self.crop_bbox = None
        self._log('info', 'Crop resetado')

### Método `CaptureManager.set_inspect_after_capture`
O que faz este método? (responda abaixo)


In [ ]:
def set_inspect_after_capture(self, inspection_type: Optional[str]=None):
        self._inspect_after_capture = inspection_type
        if inspection_type:
            self._log('debug', f'Inspeção após captura: {inspection_type}')

### Método `CaptureManager.process_captured_image`
O que faz este método? (responda abaixo)


In [ ]:
def process_captured_image(self, capture_result: Dict[str, Any]) -> Optional[Dict[str, Any]]:
        try:
            raw_image = capture_result.get('image')
            camera_info = capture_result.get('camera_info', {})
            timestamp = capture_result.get('timestamp', '')
            if raw_image is None:
                raise Exception('Nenhuma imagem no resultado de captura')
            self.raw_image = raw_image
            crop_applied = False
            processed_image = raw_image
            if self.crop_enabled and self.crop_bbox:
                try:
                    ops = [{'name': 'crop', 'bbox': self.crop_bbox}]
                    processed_image = self.core.preprocess_image(raw_image, ops)
                    crop_applied = True
                    self._log('debug', f'Crop aplicado: {self.crop_bbox}')
                except Exception as e:
                    self._log('error', f'Falha ao aplicar crop: {e}')
                    raise Exception(f'Falha ao aplicar crop: {e}')
            self.current_image = processed_image
            result = {'image': processed_image, 'raw_image': raw_image, 'camera_info': camera_info, 'timestamp': timestamp, 'shape': processed_image.shape, 'crop_applied': crop_applied}
            self._notify('capture_completed', {'shape': processed_image.shape, 'crop_applied': crop_applied})
            self._log('info', f'Imagem processada: {processed_image.shape}')
            return result
        except Exception as e:
            self._log('error', f'Erro ao processar imagem: {e}')
            self._notify('capture_error', {'error': str(e)})
            return None

### Método `CaptureManager.get_current_image`
O que faz este método? (responda abaixo)


In [ ]:
def get_current_image(self) -> Optional[np.ndarray]:
        return self.current_image

### Método `CaptureManager.get_raw_image`
O que faz este método? (responda abaixo)


In [ ]:
def get_raw_image(self) -> Optional[np.ndarray]:
        return self.raw_image

### Método `CaptureManager.get_image_info`
O que faz este método? (responda abaixo)


In [ ]:
def get_image_info(self) -> Dict[str, Any]:
        if self.current_image is None:
            return {'shape': None, 'dtype': None, 'size_mb': 0, 'crop_applied': False, 'status': 'Nenhuma imagem'}
        shape = self.current_image.shape
        dtype = str(self.current_image.dtype)
        size_mb = self.current_image.nbytes / (1024 * 1024)
        return {'shape': shape, 'dtype': dtype, 'size_mb': size_mb, 'crop_applied': self.crop_enabled, 'crop_bbox': self.crop_bbox if self.crop_enabled else None, 'status': 'OK'}

### Método `CaptureManager.clear_images`
O que faz este método? (responda abaixo)


In [ ]:
def clear_images(self):
        self.current_image = None
        self.raw_image = None
        self._inspect_after_capture = None
        self._log('debug', 'Imagens limpas')

### Método `CaptureManager.should_inspect_after_capture`
O que faz este método? (responda abaixo)


In [ ]:
def should_inspect_after_capture(self) -> Optional[str]:
        return self._inspect_after_capture

### Método `CaptureManager.apply_profile`
O que faz este método? (responda abaixo)


In [ ]:
def apply_profile(self, profile_name: str) -> bool:
        if not self.camera_profiles:
            self._log('warning', 'Nenhum camera_profiles disponível')
            return False
        try:
            if hasattr(self.camera_profiles, 'load_profile'):
                profile = self.camera_profiles.load_profile(profile_name)
            else:
                profile = getattr(self.camera_profiles, 'get_profile', lambda n: None)(profile_name)
            if not profile:
                self._log('error', f'Perfil não encontrado: {profile_name}')
                return False
            params = profile.get('parameters', {})
            if not isinstance(params, dict):
                self._log('warning', 'Parâmetros do perfil mal formatados')
                params = {}
            camera_obj = None
            if hasattr(self.core, 'camera_manager'):
                cam_mgr = self.core.camera_manager
                camera_obj = getattr(cam_mgr, 'active_camera', None) or getattr(cam_mgr, 'camera', None)
            applied_any = False
            applied_list = []
            skipped_list = []

            def normalize(name: str) -> str:
                base = name.split('@')[0]
                if '/' in base:
                    base = base.split('/')[-1]
                import re
                s1 = re.sub('(.)([A-Z][a-z]+)', '\\1_\\2', base)
                snake = re.sub('([a-z0-9])([A-Z])', '\\1_\\2', s1).lower()
                return snake

            def parse_selectors(param_key: str) -> tuple[str, dict]:
                if '@' not in param_key:
                    return (param_key, {})
                base, selector_part = param_key.split('@', 1)
                selectors = {}
                if selector_part.startswith('{') and selector_part.endswith('}'):
                    content = selector_part[1:-1]
                    for pair in content.split(','):
                        if '=' in pair:
                            key, val = pair.split('=', 1)
                            selectors[key.strip()] = val.strip()
                return (base, selectors)
            mapping = {'acquisition_frame_rate': 'frame_rate', 'balancewhiteauto': 'balance_white', 'balance_white_auto': 'balance_white'}
            if isinstance(camera_obj, object) and hasattr(camera_obj, 'apply_pfs_file') and profile.get('_pfs_file'):
                pfs_path = profile.get('_pfs_file')
                self._log('debug', f'Delegando aplicação .pfs direto para câmera: {pfs_path}')
                try:
                    result = camera_obj.apply_pfs_file(pfs_path)
                    return bool(result)
                except Exception as e:
                    self._log('error', f'Falha ao aplicar .pfs via câmera: {e}')
            for key, val in params.items():
                if key in ['format', 'metadata', 'file_info', 'converted_at']:
                    continue
                param_name, selectors = parse_selectors(key)
                norm = normalize(param_name)
                norm = mapping.get(norm, norm)
                selector_applied_ok = True
                for sel_key in sorted(selectors.keys()):
                    sel_val = selectors[sel_key]
                    sel_norm = normalize(sel_key)
                    if camera_obj and hasattr(camera_obj, 'set_parameter'):
                        try:
                            success = camera_obj.set_parameter(sel_norm, str(sel_val))
                            self._log('debug', f'Seletor {sel_norm}={sel_val} -> {success}')
                            if not success:
                                selector_applied_ok = False
                                self._log('debug', f'Seletor {sel_norm} não pôde ser aplicado')
                        except Exception as e:
                            self._log('debug', f'Erro ao aplicar seletor {sel_norm}: {e}')
                            selector_applied_ok = False
                if not selector_applied_ok and selectors:
                    self._log('debug', f'Pulando {norm} pois seletores falharam')
                    skipped_list.append(norm)
                    continue
                if camera_obj and hasattr(camera_obj, 'set_parameter'):
                    try:
                        success = camera_obj.set_parameter(norm, str(val))
                        self._log('debug', f'Aplicando {norm}={val} -> {success}')
                        if success:
                            applied_any = True
                            applied_list.append(f'{norm}={val}')
                        else:
                            skipped_list.append(norm)
                    except Exception as e:
                        self._log('error', f'Erro ajustando parâmetro {norm}: {e}')
                        skipped_list.append(norm)
                else:
                    self._log('warning', f'Nenhuma câmera ou método set_parameter não disponível para {norm}')
                    skipped_list.append(norm)
            if applied_any:
                self._log('info', f"Perfil '{profile_name}' aplicado: {len(applied_list)} parâmetro(s) ajustado(s)")
                self._log('debug', f'Parâmetros aplicados: {applied_list}')
            else:
                self._log('warning', f"Perfil '{profile_name}' não alterou nenhum parâmetro")
            if skipped_list:
                self._log('debug', f'Parâmetros pulados/inválidos: {skipped_list}')
            return applied_any
        except Exception as e:
            self._log('error', f'Erro ao aplicar perfil: {e}')
            return False

## Classe `HistoryManager`
O que representa esta classe? (responda abaixo)


### Método `HistoryManager.__init__`
O que faz este método? (responda abaixo)


In [ ]:
def __init__(self, core, results_dir: Optional[Path]=None):
        super().__init__(core)
        self.results_dir = Path(results_dir) if results_dir else Path('data/results')
        self.loaded_history: List[Dict] = []
        self._log('info', f'Inicializado com diretório: {self.results_dir}')

### Método `HistoryManager.initialize`
O que faz este método? (responda abaixo)


In [ ]:
def initialize(self) -> bool:
        try:
            self.load_history()
            return True
        except Exception as e:
            self._log('error', f'Falha ao inicializar: {e}')
            return False

### Método `HistoryManager.cleanup`
O que faz este método? (responda abaixo)


In [ ]:
def cleanup(self) -> None:
        self.loaded_history.clear()
        self._log('info', 'Limpeza completa')

### Método `HistoryManager.load_history`
O que faz este método? (responda abaixo)


In [ ]:
def load_history(self) -> List[Dict]:
        try:
            self.loaded_history.clear()
            if not self.results_dir.exists():
                self._log('warning', f'Diretório não existe: {self.results_dir}')
                return []
            history = []
            for result_dir in sorted(self.results_dir.iterdir(), reverse=True):
                if not result_dir.is_dir():
                    continue
                inspection_file = result_dir / 'inspection_data.json'
                if inspection_file.exists():
                    try:
                        with open(inspection_file, 'r', encoding='utf-8') as f:
                            data = json.load(f)
                        history.append({'timestamp': data.get('timestamp'), 'inspection_type': data.get('inspection_type'), 'model_name': data.get('model_name'), 'path': str(result_dir), 'data': data.get('results', {})})
                    except Exception as e:
                        self._log('warning', f'Falha ao carregar {inspection_file}: {e}')
            self.loaded_history = history
            self._notify('history_loaded', {'count': len(history)})
            self._log('info', f'Histórico carregado: {len(history)} inspeções')
            return history
        except Exception as e:
            self._log('error', f'Erro ao carregar histórico: {e}')
            self._notify('history_error', {'error': str(e)})
            return []

### Método `HistoryManager.get_history_item`
O que faz este método? (responda abaixo)


In [ ]:
def get_history_item(self, index: int) -> Optional[Dict]:
        if 0 <= index < len(self.loaded_history):
            return self.loaded_history[index]
        return None

### Método `HistoryManager.get_history_by_timestamp`
O que faz este método? (responda abaixo)


In [ ]:
def get_history_by_timestamp(self, timestamp: str) -> Optional[Dict]:
        for item in self.loaded_history:
            if item['timestamp'] == timestamp:
                return item
        return None

### Método `HistoryManager.get_history_count`
O que faz este método? (responda abaixo)


In [ ]:
def get_history_count(self) -> int:
        return len(self.loaded_history)

### Método `HistoryManager.view_history_item`
O que faz este método? (responda abaixo)


In [ ]:
def view_history_item(self, item_index: int) -> Optional[str]:
        item = self.get_history_item(item_index)
        if not item:
            return None
        lines = []
        lines.append('=' * 60)
        lines.append(f"Data/Hora: {item.get('timestamp', 'N/A')}")
        lines.append(f"Tipo: {item.get('inspection_type', 'N/A').upper()}")
        lines.append(f"Modelo: {item.get('model_name', 'N/A')}")
        lines.append('=' * 60)
        results = item.get('data', {})
        if item.get('inspection_type') == 'segmentation':
            lines.append(f"Classe: {results.get('class', 'N/A')}")
            lines.append(f"Confiança: {results.get('confidence', 0):.2%}")
            lines.append(f"Área: {results.get('area', 'N/A')}")
        elif item.get('inspection_type') == 'classification':
            lines.append(f"Classe Predita: {results.get('predicted_class', 'N/A')}")
            lines.append(f"Confiança: {results.get('confidence', 0):.2%}")
            if 'class_confidences' in results:
                lines.append('\nConfiança por classe:')
                for cls, conf in results['class_confidences'].items():
                    lines.append(f'  - {cls}: {conf:.2%}')
        lines.append('=' * 60)
        return '\n'.join(lines)

### Método `HistoryManager.export_to_csv`
O que faz este método? (responda abaixo)


In [ ]:
def export_to_csv(self, output_path: Path) -> bool:
        try:
            if not self.loaded_history:
                self._log('warning', 'Histórico vazio para exportar')
                return False
            with open(output_path, 'w', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow(['Timestamp', 'Tipo', 'Modelo', 'Resultado', 'Confiança'])
                for item in self.loaded_history:
                    timestamp = item.get('timestamp', '')
                    inspection_type = item.get('inspection_type', '')
                    model_name = item.get('model_name', '')
                    data = item.get('data', {})
                    if inspection_type == 'segmentation':
                        resultado = data.get('class', 'N/A')
                    else:
                        resultado = data.get('predicted_class', 'N/A')
                    confianca = f"{data.get('confidence', 0):.2%}"
                    writer.writerow([timestamp, inspection_type, model_name, resultado, confianca])
            self._log('info', f'Histórico exportado para: {output_path}')
            self._notify('history_exported', {'path': str(output_path)})
            return True
        except Exception as e:
            self._log('error', f'Erro ao exportar CSV: {e}')
            self._notify('history_error', {'error': str(e)})
            return False

### Método `HistoryManager.delete_history_item`
O que faz este método? (responda abaixo)


In [ ]:
def delete_history_item(self, timestamp: str) -> bool:
        try:
            result_dir = self.results_dir / timestamp
            if result_dir.exists():
                shutil.rmtree(result_dir)
                self._log('info', f'Deletado diretório: {result_dir}')
            original_count = len(self.loaded_history)
            self.loaded_history = [item for item in self.loaded_history if item['timestamp'] != timestamp]
            deleted = original_count - len(self.loaded_history)
            if deleted > 0:
                self._notify('history_modified', {'deleted': deleted})
                self._log('info', f'Item {timestamp} removido do histórico')
                return True
            return False
        except Exception as e:
            self._log('error', f'Erro ao deletar histórico: {e}')
            self._notify('history_error', {'error': str(e)})
            return False

### Método `HistoryManager.clear_all_history`
O que faz este método? (responda abaixo)


In [ ]:
def clear_all_history(self) -> bool:
        try:
            if self.results_dir.exists():
                shutil.rmtree(self.results_dir)
                self._log('warning', 'Histórico completamente deletado do disco')
            self.results_dir.mkdir(parents=True, exist_ok=True)
            self.loaded_history.clear()
            self._notify('history_modified', {'cleared': True})
            self._log('warning', 'Todo o histórico foi limpo')
            return True
        except Exception as e:
            self._log('error', f'Erro ao limpar histórico: {e}')
            self._notify('history_error', {'error': str(e)})
            return False

### Método `HistoryManager.get_summary_stats`
O que faz este método? (responda abaixo)


In [ ]:
def get_summary_stats(self) -> Dict[str, Any]:
        if not self.loaded_history:
            return {'total': 0, 'segmentation': 0, 'classification': 0, 'models': {}}
        segmentation_count = sum((1 for item in self.loaded_history if item.get('inspection_type') == 'segmentation'))
        classification_count = sum((1 for item in self.loaded_history if item.get('inspection_type') == 'classification'))
        models = {}
        for item in self.loaded_history:
            model = item.get('model_name', 'Unknown')
            models[model] = models.get(model, 0) + 1
        return {'total': len(self.loaded_history), 'segmentation': segmentation_count, 'classification': classification_count, 'models': models}